### Why this exists

A chunk store answers "which 400 characters mention this". That is the wrong unit for half of the
questions people actually ask a corpus:

| question | what the chunk store gives you | what it should give you |
|---|---|---|
| "where in this 500-page book is X discussed?" | five fragments | a chapter |
| "summarise section 3" | fragments that mention section 3 | section 3 |
| "fill this template from these reports" | fragments | cited sections per question |

The fix is not a better chunker. It is remembering **where each chunk lives**. `litesearch.tree`
builds a [PageIndex](https://github.com/VectifyAI/PageIndex)-style node tree per document at
ingest time, links every chunk to a node, and adds three things on top of the existing hybrid
search: evidence that **rolls up** to sections, `toc()`/`read()` for reasoning over structure with
no embeddings at all, and chunk **spans** so a generating model sees contiguous text.

Nothing here calls a language model. PageIndex builds its tree with one; the structural signal —
markdown headings, then chapter lines, then page windows as the floor — gets most of the value at
zero token cost, and every hook that *would* want a model (`summarize=`, `build=`) takes a
callable, so one can be dropped in exactly where it pays.

> Adapted from `chitragupta`, which prototyped this over an astrology-book library. What lives here
> is the part that has nothing to do with any particular corpus.

In [ ]:
#| default_exp tree

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from fastcore.all import Path, patch, first, L, AttrDict, ifnone
from fastlite import Database
from apswutils.db import Table
from apswutils.utils import hash_record
from dataclasses import dataclass, field
import re
import numpy as np

from litesearch.core import _in, _rid, rrf_merge, process_content
from litesearch.data import chunk_markdown

## Detecting structure

Three signals, tried in order. Markdown headings are the strongest and the most common (anything
that went through `pdf_markdown`, a docs site, or a notebook has them). Classic books that were
scanned to plain text have chapter lines instead. Everything else falls back to fixed page
windows, which are a poor tree but keep the invariant that *every* document is navigable — a
`toc()` that sometimes returns nothing is a `toc()` nobody calls.

In [ ]:
#| export
_md_head  = re.compile(r'^(#{1,6})\s+(.+?)\s*#*\s*$')
# A structural heading line, with an optional leading `#` — a PDF converter may already have
# marked it up, and the word is a better signal than the markup either way.
_STRUCT = ('BOOK','TITLE','PART','ANNEX','APPENDIX','CANTO','CHAPTER','ADHYAYA','SECTION','SUBTITLE',
           'ARTICLE','RULE','CLAUSE','LESSON','SCHEDULE')
_chapter  = re.compile(r'^\s{0,6}#{0,6}\s*(' + '|'.join(_STRUCT) + r')\s+([IVXLCDM]+|\d+[A-Za-z]?)'
                       r'\b[\s.:—-]*(.{0,80})$', re.I)
# Prior order of the usual hierarchy words. Lower sits higher in the tree. Anything not listed is
# ranked by how often it occurs, since a rarer heading word names a bigger division.
_STRUCT_RANK = {'book':0, 'title':0, 'part':0, 'annex':0, 'appendix':0, 'canto':0,
                'chapter':1, 'adhyaya':1, 'section':2, 'subtitle':2,
                'article':3, 'rule':3, 'clause':3, 'lesson':3, 'schedule':3}
# Above this many markdown headings per page, `#` has stopped meaning "heading" — see `detect_mode`.
MAX_HEAD_DENSITY = 4.0
_ws       = re.compile(r'\s+')

@dataclass
class TreeNode:
    'One section of a document. `seq` is its index in the flat list; `parent` is another `seq`.'
    seq:int
    title:str
    level:int
    parent:int|None = None
    page_start:int = 0
    page_end:int = 0
    segments:list = field(default_factory=list)   # [(page, text), ...]
    children:list = field(default_factory=list)   # [seq, ...]
    summary:str = ''
    def text(self): return '\n\n'.join(s for _, s in self.segments if s.strip())

def summarize_extractive(text:str, n:int=300) -> str:
    'First `n` clean characters. Cheap, deterministic, and the seam an LLM slots into via `summarize=`.'
    t = _ws.sub(' ', text).strip()
    return t[:n] + ('…' if len(t) > n else '')

def _clean_title(t:str, max_len:int=100): return _ws.sub(' ', t).strip(' #*_')[:max_len].strip()

def struct_levels(pages, max_levels:int=4) -> dict:
    '''`{heading word: level}` for one document, as consecutive levels starting at 1.

    Levels are compacted rather than fixed, so a directive that only ever says "Article" gets
    articles at level 1 instead of burying every one of them four deep. Unknown words are ranked
    by frequency: the rarer heading word names the bigger division.'''
    seen = {}
    for _, txt in pages:
        for ln in (txt or '').splitlines():
            if (m := _chapter.match(ln)): seen[m.group(1).lower()] = seen.get(m.group(1).lower(), 0) + 1
    if not seen: return {}
    unknown = sorted((n, w) for w, n in seen.items() if w not in _STRUCT_RANK)
    rank = {w: (_STRUCT_RANK[w], 0) for w in seen if w in _STRUCT_RANK}
    rank |= {w: (99, i) for i, (_, w) in enumerate(unknown)}
    order = sorted({rank[w] for w in seen})
    return {w: min(order.index(rank[w]) + 1, max_levels) for w in seen}

def _md_stats(pages):
    'Markdown heading count and how many distinct `#` depths appear.'
    n, lvls = 0, set()
    for _, txt in pages:
        for ln in (txt or '').splitlines():
            if (m := _md_head.match(ln)): n += 1; lvls.add(len(m.group(1)))
    return n, len(lvls)

def detect_mode(pages) -> str:
    '''Which structural signal this document carries: `markdown`, `chapter` or `window`.

    Two thresholds, because `#` fails in both directions. Too few headings and a 600-page scan is
    being judged on three stray hashes, so the bar rises with length. Too *many* — a PDF converter
    that marks every bold line as an h1, which is the common case for legislation and reports —
    and `#` has stopped carrying structure at all: the tell is a high density with no variation in
    depth, and the words in the text (CHAPTER, Article) are then the better signal.'''
    md, depths = _md_stats(pages)
    ch = sum(1 for _, txt in pages for ln in (txt or '').splitlines() if _chapter.match(ln))
    npg = max(1, len(pages))
    if md/npg > MAX_HEAD_DENSITY and depths < 2:
        return 'chapter' if ch >= 3 else 'window'
    # short docs are usually born-markdown (a README, a note); long ones are usually scans where a
    # stray `#` is punctuation, so the bar rises with length instead of being one fixed number
    if md >= (2 if len(pages) <= 3 else max(3, npg // 25)): return 'markdown'
    return 'chapter' if ch >= 3 else 'window' 

`build_tree` returns a **flat list** rather than a nested one: `nodes[seq]` is O(1), the parent
link is an int, and it maps onto a SQL table without a serialiser. Nesting is a view over it
(`toc()` builds one on demand).

In [ ]:
#| export
def build_tree(pages,                  # [(page_no, text)] — markdown or plain text
               title:str='Document',   # title of the root node
               summarize=None,         # callable(text)->str for node summaries (LLM goes here)
               window:int=8,           # pages per node in `window` mode
               min_level:int=1,        # floor for heading depth
               max_levels:int=4        # deepest node level kept
) -> list:
    'A flat list of `TreeNode` for a document (index = seq, root = 0).'
    pages = [(p, t or '') for p, t in pages]
    summarize = summarize or summarize_extractive
    mode, nodes = detect_mode(pages), []
    slev = struct_levels(pages, max_levels) if mode == 'chapter' else {}
    def fresh(t, lvl, parent, page):
        nd = TreeNode(seq=len(nodes), title=_clean_title(t) or f'Section {len(nodes)}',
                      level=lvl, parent=parent, page_start=page, page_end=page)
        nodes.append(nd)
        if parent is not None: nodes[parent].children.append(nd.seq)
        return nd
    root = fresh(title, 0, None, pages[0][0] if pages else 0)
    stack, cur, buf, cur_page = [root], root, [], root.page_start
    def open_node(t, lvl, page):
        nonlocal cur
        lvl = min(lvl, max_levels)
        while len(stack) > lvl: stack.pop()
        cur = fresh(t, lvl, stack[-1].seq, page)
        stack.append(cur)
    def flush(page):
        if (txt := '\n'.join(buf).strip()): cur.segments.append((page, txt))
        cur.page_end = max(cur.page_end, page)
        buf.clear()

    if mode == 'window':
        for i in range(0, len(pages), window):
            grp = pages[i:i+window]
            lead = next((l.strip() for _, t in grp for l in t.splitlines() if l.strip()), '')
            open_node(f'Pages {grp[0][0]+1}–{grp[-1][0]+1}: {lead[:60]}', 1, grp[0][0])
            for p, t in grp:
                if t.strip(): cur.segments.append((p, t.strip()))
                cur.page_end = p
        cur = root
    else:
        for p, txt in pages:
            for ln in txt.splitlines():
                m = _md_head.match(ln) if mode == 'markdown' else _chapter.match(ln)
                # a lowercase first character usually means a `#` that was punctuation, not a heading
                if m and (mode == 'chapter' or (len(t := _clean_title(m.group(2))) >= 2 and not t[0].islower())):
                    flush(cur_page)
                    open_node(ln.strip() if mode == 'chapter' else m.group(2),
                              slev.get(m.group(1).lower(), 1) if mode == 'chapter'
                              else max(min_level, len(m.group(1))), p)
                    cur_page = p
                    continue
                buf.append(ln)
            flush(p)
            cur_page = p
    flush(cur_page)
    for nd in nodes:
        base = nd.text() or ' / '.join(nodes[c].title for c in nd.children[:8])
        nd.summary = summarize(base) if base else ''
    root.page_end = max((n.page_end for n in nodes), default=root.page_end)
    return nodes

def heading_path(tree,              # the node list from build_tree
                 nd,                # the TreeNode to describe
                 title:str,         # document title
                 sep:str=' › ',
                 max_len:int=200) -> str:
    '''`Doc › Part › Chapter` for one node.

    Embedded with the chunk and indexed for FTS, this is what stops a chunk reading as an
    out-of-context fragment: "the effects are severe" means nothing until you know which chapter
    said it.'''
    parts, cur = [], nd
    while cur is not None:
        if cur.level > 0: parts.append(cur.title)
        cur = tree[cur.parent] if cur.parent is not None else None
    return sep.join([title] + parts[::-1])[:max_len]

In [ ]:
tree = build_tree([(0, '# Saturn\n\nIntro text about the ringed planet.\n\n## Transits\n\nSade sati runs seven years.'),
                   (1, '## Remedies\n\nRecite on Saturdays.')], title='Jyotisha')
for n in tree: print(f'{n.seq}  lvl{n.level}  p{n.page_start}-{n.page_end}  {n.title!r:24} {n.summary[:40]!r}')
assert [n.title for n in tree] == ['Jyotisha', 'Saturn', 'Transits', 'Remedies']
assert tree[2].parent == 1 and tree[3].parent == 1, 'h2s hang off the h1, not off each other'
assert heading_path(tree, tree[2], 'Jyotisha') == 'Jyotisha › Saturn › Transits'

0  lvl0  p0-1  'Jyotisha'               'Saturn'
1  lvl1  p0-0  'Saturn'                 'Intro text about the ringed planet.'
2  lvl2  p0-0  'Transits'               'Sade sati runs seven years.'
3  lvl2  p1-1  'Remedies'               'Recite on Saturdays.'


In [ ]:
# a scanned book has chapter lines, not markdown; and a doc with neither still gets a tree
_book = build_tree([(i, f'CHAPTER {i+1}\n\nbody of chapter {i+1}') for i in range(4)], title='Saravali')
assert detect_mode([(i, f'CHAPTER {i+1}\n\nbody') for i in range(4)]) == 'chapter'
assert len(_book) == 5 and _book[1].title.startswith('CHAPTER 1')

_plain = build_tree([(i, f'page {i} of undifferentiated prose') for i in range(20)], title='Notes', window=8)
assert detect_mode([(i, 'prose') for i in range(20)]) == 'window'
assert len(_plain) == 4, [n.title for n in _plain]   # root + ceil(20/8) windows
assert _plain[1].title.startswith('Pages 1–8')

## The tables

`get_tree` sits beside `get_store` the way `get_graph` does: two extra tables and three extra
columns on the chunk store. The store stays the source of truth and everything else in litesearch
— FTS5, the ANN index, `db.search`, the graph layer — keeps working unchanged on it.

```
docs    id · title · source · kind · pages · meta · added_at
nodes   id ('doc#seq') · doc_id · parent_id · level · seq · title ·
        page_start · page_end · summary · nchunks
store   content · embedding · metadata · doc_id · node_id · page · heading   [+ FTS5, +ANN]
```

Doc ids are content-addressed on `(source, title)`, so re-adding a document is a no-op rather than
a duplicate — the same property `hash=True` gives chunks.

In [ ]:
#| export
def doc_id(source, title='') -> str:
    'Content-addressed document id — re-adding the same source is a no-op, not a duplicate.'
    return hash_record({'k': f'{source}|{title}'})[:16]

@patch
def get_tree(self:Database,
             store:str='store',   # chunk store the tree is built over
             prefix:str=None,     # table prefix (default: '' for 'store', else '<store>_')
             ann:bool=True,       # register an ANN index on the chunk store
             **kw                 # extra typed columns for the chunk store
) -> AttrDict:
    'Create the docs/nodes tables and a node-aware chunk store. Idempotent; returns the tables.'
    p = prefix if prefix is not None else ('' if store == 'store' else f'{store}_')
    dt, nt = f'{p}docs', f'{p}nodes'
    st = self.get_store(store, hash=True, ann=ann, doc_id=str, node_id=str, page=int, heading=str, **kw)
    self.t[dt].create(id=str, title=str, source=str, kind=str, pages=int, meta=str, added_at=float,
                      pk='id', if_not_exists=True, defaults=dict(added_at='CURRENT_TIMESTAMP'))
    self.t[nt].create(id=str, doc_id=str, parent_id=str, level=int, seq=int, title=str,
                      page_start=int, page_end=int, summary=str, nchunks=int, pk='id', if_not_exists=True)
    for t, c in ((nt,'doc_id'), (nt,'parent_id'), (store,'doc_id'), (store,'node_id')):
        self.t[t].create_index([c], if_not_exists=True)
    return AttrDict(docs=self.t[dt], nodes=self.t[nt], store=st, prefix=p)

## Ingestion

`add_doc` is the whole pipeline: tree → node-aware chunks → embed → store. It takes `pages` rather
than a file, so acquisition stays someone else's problem — `litesearch.data.pdf_parse` for PDFs,
`file_parse` for anything on disk, fossick for the web. The one thing it insists on is that a
chunk carries its `heading` path, both embedded with the text and stored for FTS.

In [ ]:
#| export
MIN_CHUNK = 40

def _node_chunks(tree, title, did, chunker=None, min_chunk=MIN_CHUNK):
    """Chunk every node segment, tagging each chunk with its node, page and heading path.

    A chunk under `min_chunk` is *merged into its predecessor*, never dropped. Dropping is the
    tempting reading of a minimum size and it is wrong here: `read()` assembles a section out of
    its chunks, so a discarded chunk is text that has silently left the corpus."""
    out = L()
    for nd in tree:
        head, nid = heading_path(tree, nd, title), f'{did}#{nd.seq}'
        for page, seg in nd.segments:
            prev = None
            for c in chunk_markdown(seg, chunker):
                if not (c or '').strip(): continue
                if prev is not None and len(c.strip()) < min_chunk:
                    prev['content'] = f"{prev['content']}\n\n{c}"
                    continue
                prev = dict(content=c, doc_id=did, node_id=nid, page=page, heading=head)
                out.append(prev)
    return out

@patch
def add_doc(self:Database,
            pages,                  # [(page_no, text)] — or a single string
            title:str,              # document title
            source:str=None,        # path or url (defaults to the title)
            kind:str='text',        # 'pdf' | 'web' | 'md' | 'code' | anything you filter on
            store:str='store',      # chunk store
            prefix:str=None,        # tree table prefix
            emb_fn=None,            # embedder: list[str] -> vectors
            chunker=None,           # chonkie chunker (default: FastChunker via chunk_markdown)
            summarize=None,         # callable(text)->str for node summaries
            with_heading:bool=True, # embed each chunk together with its heading path
            meta:dict=None,         # arbitrary json metadata for the doc row
            force:bool=False        # re-ingest a document already present
) -> dict:
    '''Ingest one document: build its tree, chunk it per node, embed and store.

    Chunks are embedded as `heading ⏎⏎ content` but **stored** as bare content, so retrieval sees
    the context and the caller gets clean text back.'''
    import json as _json
    if isinstance(pages, str): pages = [(0, pages)]
    pages = [(p, t or '') for p, t in pages]
    g, src = self.get_tree(store, prefix), str(ifnone(source, title))
    did = doc_id(src, title)
    if first(g.docs(where=f'id={did!r}')):
        if not force: return dict(doc_id=did, title=title, skipped='already ingested; pass force=True')
        self.delete_doc(did, store, prefix)
    tree = build_tree(pages, title=title, summarize=summarize)
    g.docs.insert(dict(id=did, title=title, source=src, kind=kind,
                       pages=(max(p for p, _ in pages)+1 if pages else 0),
                       meta=_json.dumps(meta or {})), replace=True)
    chunks = _node_chunks(tree, title, did, chunker)
    counts = {}
    for c in chunks: counts[c['node_id']] = counts.get(c['node_id'], 0) + 1
    g.nodes.insert_all([dict(id=f'{did}#{nd.seq}', doc_id=did, seq=nd.seq, level=nd.level,
                             parent_id=None if nd.parent is None else f'{did}#{nd.parent}',
                             title=nd.title, page_start=nd.page_start, page_end=nd.page_end,
                             summary=nd.summary, nchunks=counts.get(f'{did}#{nd.seq}', 0))
                        for nd in tree], replace=True)
    if chunks:
        fn = emb_fn if not (emb_fn and with_heading) else (
            lambda ts, **kw: emb_fn([f"{h}\n\n{t}" if (h := heads.get(t)) else t for t in ts], **kw))
        heads = {c['content']: c['heading'] for c in chunks}
        process_content(g.store, list(chunks), embed=bool(emb_fn), emb_fn=fn)
        if self._ann_meta(store): g.store.rebuild_index()
    return dict(doc_id=did, title=title, kind=kind, nodes=len(tree), chunks=len(chunks))

@patch
def delete_doc(self:Database, did:str, store:str='store', prefix:str=None):
    'Remove a document, its nodes and its chunks; rebuild the ANN index so no keys are orphaned.'
    g = self.get_tree(store, prefix)
    g.store.delete_where(where=f'doc_id={did!r}')
    g.nodes.delete_where(where=f'doc_id={did!r}')
    g.docs.delete_where(where=f'id={did!r}')
    if self._ann_meta(store): g.store.rebuild_index()

`add_file` is the one-liner for the common case. It only knows about *documents* — PDFs, markdown,
plain text, notebooks. Source code is a different tree (module › class › function) and belongs to
`kosha`, which builds it from the AST rather than from headings.

In [ ]:
#| export
DOC_EXTS = '.pdf,.md,.markdown,.txt,.rst,.ipynb'

@patch
def add_file(self:Database,
             path,                 # file to ingest
             title:str=None,       # defaults to a prettified filename
             store:str='store',
             prefix:str=None,
             **kw                  # forwarded to add_doc (emb_fn, chunker, summarize, force, ...)
) -> dict:
    'Ingest one document file. PDFs page through `pdf_parse`; everything else is one page of text.'
    p = Path(path)
    ttl = title or p.stem.replace('_',' ').replace('-',' ').strip()
    if p.suffix.lower() == '.pdf':
        from litesearch.data import pdf_parse
        pages, kind = list(enumerate(pdf_parse(str(p)))), 'pdf'
    elif p.suffix.lower() == '.ipynb':
        from litesearch.data import ipynb_parse
        pages, kind = [(0, '\n\n'.join(c['content'] for c in ipynb_parse(p)))], 'notebook'
    else:
        pages, kind = [(0, p.read_text(errors='replace'))], p.suffix.lstrip('.').lower() or 'text'
    return self.add_doc(pages, ttl, source=str(p), kind=kind, store=store, prefix=prefix, **kw)

@patch
def add_dir(self:Database,
            dir,                    # directory to walk
            types:str=DOC_EXTS,     # comma-separated extensions to ingest
            store:str='store',
            prefix:str=None,
            **kw                    # forwarded to add_doc
) -> list:
    'Ingest every document under a directory tree. Already-ingested sources are skipped, not duplicated.'
    exts = {f".{t.strip().lstrip('.')}".lower() for t in types.split(',')}
    return [self.add_file(p, store=store, prefix=prefix, **kw)
            for p in sorted(Path(dir).rglob('*')) if p.is_file() and p.suffix.lower() in exts]

## Reading the structure

`toc` and `read` are the vectorless half — PageIndex's actual argument. An agent that already
knows it wants "the chapter on remedies" should not be asked to phrase that as a similarity query;
it should read the table of contents and open the node. No embedding is computed by either call.

In [ ]:
#| export
@patch
def toc(self:Database,
        doc:str=None,           # doc id, or a substring of the title (None = every document)
        store:str='store',
        prefix:str=None,
        max_depth:int=3,        # deepest level included
        summaries:bool=True     # include node summaries
) -> list:
    'The document tree as nested dicts — titles, page ranges, summaries. No embeddings touched.'
    g = self.get_tree(store, prefix)
    docs = (g.docs(where=f'id={doc!r} or title like {"%"+doc+"%"!r}') if doc else g.docs())
    out = []
    for d in docs:
        rows = sorted(g.nodes(where=f'doc_id={d["id"]!r}'), key=lambda r: r['seq'])
        kids = {}
        for r in rows: kids.setdefault(r['parent_id'], []).append(r)
        def node(r):
            o = dict(id=r['id'], title=r['title'], level=r['level'], nchunks=r['nchunks'],
                     pages=(r['page_start'], r['page_end']))
            if summaries and r['summary']: o['summary'] = r['summary']
            if r['level'] < max_depth and (ch := kids.get(r['id'])): o['children'] = [node(c) for c in ch]
            return o
        root = first(rows, lambda r: r['parent_id'] is None)
        out.append(dict(doc_id=d['id'], title=d['title'], source=d['source'], pages=d['pages'],
                        tree=node(root) if root else None))
    return out

@patch
def breadcrumb(self:Database, node_id:str, store:str='store', prefix:str=None, sep:str=' › ') -> str:
    'The path from the document down to a node: `Book › Chapter › Section`.'
    g = self.get_tree(store, prefix)
    parts, cur, seen = [], first(g.nodes(where=f'id={node_id!r}')), set()
    while cur and cur['id'] not in seen:
        seen.add(cur['id'])
        parts.append(cur['title'])
        cur = first(g.nodes(where=f'id={cur["parent_id"]!r}')) if cur['parent_id'] else None
    return sep.join(parts[::-1])

@patch
def read(self:Database,
         node_id:str,           # 'doc#seq', as returned by toc() or sections()
         store:str='store',
         prefix:str=None,
         max_chars:int=20000,   # cap on the assembled text
         children:bool=True     # append the text of child nodes too
) -> dict:
    'One whole section — the unit an agent should read instead of guessing from fragments.'
    g = self.get_tree(store, prefix)
    nd = first(g.nodes(where=f'id={node_id!r}'))
    if not nd: return {}
    ids = [node_id]
    if children:
        frontier = [node_id]
        while frontier:
            kids = [r['id'] for r in g.nodes(where=_in('parent_id', frontier))]
            if not kids: break
            ids += kids
            frontier = kids
    rows = sorted(g.store(select='content, node_id, page', where=_in('node_id', ids)),
                  key=lambda r: (ids.index(r['node_id']) if r['node_id'] in ids else 1<<30, r['page'] or 0))
    text = '\n\n'.join(r['content'] for r in rows)[:max_chars]
    return dict(id=node_id, title=nd['title'], breadcrumb=self.breadcrumb(node_id, store, prefix),
                pages=(nd['page_start'], nd['page_end']), summary=nd['summary'],
                children=[r['id'] for r in g.nodes(where=f'parent_id={node_id!r}')], text=text)

## Retrieval that knows about structure

Three refinements over `db.search`, each addressing a way a flat chunk list misleads:

**Adaptive fusion.** Plain RRF weights both legs equally on every query, which is wrong in two
directions that are cheap to detect. A quoted phrase or a name/number-heavy query ("Vimshottari",
`get_store`, "1706.03762") is a keyword query wearing a sentence's clothes; and when FTS returns
nothing at all, an equal vote just dilutes the vector ranking with noise.

**Spans.** Adjacent chunks that both hit are one passage cut in two. Merging them inside a node —
and padding with the neighbours on either side — gives a generating model contiguous text instead
of two fragments with a hole in the middle.

**Section rollup.** `sections()` groups hits by node and sums their RRF mass, so five weak hits
spread across one chapter outrank one strong hit in an unrelated appendix. Each result carries the
`read()` handle for the next call, which is the point: the agent's next action is in the payload.

In [ ]:
#| export
_QUOTED = re.compile(r'"[^"]+"')
_NAMEY  = re.compile(r'[A-Z][a-z]{2,}|\d|_')

def adaptive_weights(q:str,      # the raw query
                     fts:list,   # the FTS leg's results
                     vec:list    # the vector leg's results
) -> tuple:
    '''`(fts_weight, vec_weight)` for RRF, from cheap signals in the query and the legs.

    Three rules, not a learned model: a quoted phrase and a rare identifier are literal requests,
    an empty leg cannot vote, and everything else is a tie. **Measured inert** on prose queries —
    neither trigger fires on an ordinary sentence — so `doc_search(adaptive=...)` is off by
    default. Worth turning on only where users really do type quoted phrases.'''
    if not fts: return (0.0, 1.0)
    if not vec: return (1.0, 0.0)
    w = 1.0
    if _QUOTED.search(q or ''): w += 0.6
    toks = (q or '').split()
    if toks and sum(bool(_NAMEY.search(t)) for t in toks)/len(toks) >= 0.5: w += 0.4
    return (w, 1.0)

def merge_spans(hits,            # ranked hits carrying node_id + page
                gap:int=1        # merge hits at most this many pages apart
) -> list:
    '''Collapse hits that are adjacent inside the same node into one span.

    The merged hit keeps the best rank and score of its members, so ordering is unchanged; what
    changes is that the caller gets one contiguous passage where it had two halves.'''
    out, byn = [], {}
    for i, h in enumerate(hits):
        nid = h.get('node_id')
        if not nid: out.append(h); continue
        byn.setdefault(nid, []).append((i, h))
    for nid, group in byn.items():
        group.sort(key=lambda t: (t[1].get('page') or 0, t[0]))
        cur = None
        for i, h in group:
            if cur and (h.get('page') or 0) - (cur['_page_end'] or 0) <= gap:
                cur['content'] = f"{cur['content']}\n\n{h.get('content') or ''}"
                cur['_page_end'] = h.get('page') or cur['_page_end']
                cur['_nspan'] += 1
                cur['_rrf_score'] = max(cur.get('_rrf_score', 0), h.get('_rrf_score', 0))
                cur['_rank'] = min(cur['_rank'], i)
            else:
                cur = dict(h, _page_end=h.get('page'), _nspan=1, _rank=i)
                out.append(cur)
    return sorted(out, key=lambda h: h.get('_rank', 1<<30))

@patch
def doc_search(self:Database,
               q:str,                 # query string
               emb:bytes,             # query embedding
               columns:list=None,     # extra columns (node_id, page, heading, doc_id always included)
               limit:int=10,
               store:str='store',
               prefix:str=None,
               spans:bool=True,       # merge adjacent hits inside a node
               gap:int=1,             # span merge tolerance, in pages
               adaptive:bool=False,   # weight the RRF legs by query shape (measured inert — see below)
               rrf_k:int=60,
               **kw                   # forwarded to Database.search
) -> list:
    """Hybrid search over a node-aware store: span merging and a breadcrumb per hit.

    `adaptive` defaults **off**. Its triggers — a quoted phrase, a name/number-heavy query — did
    not fire once across 300 natural-language queries over 486 pages of legislation, giving
    results identical to plain RRF to three decimal places, and a replacement rule that leaned on
    the FTS leg's coverage measured *worse* (0.425 against 0.496 MRR on degraded queries). Kept
    and opt-in: the case it was written for — users who really do type quoted phrases and
    identifiers — is real, and simply is not what that corpus tests.
    """
    cols = list(dict.fromkeys((columns or ['content']) + ['node_id','page','heading','doc_id','rowid']))
    base = self.search(q, emb, columns=cols, limit=max(limit*3, 30), table_name=store, rrf=False, **kw)
    if not base: return []
    fts, vec = base['fts'], base['vec']
    wf, wv = adaptive_weights(q, fts, vec) if adaptive else (1.0, 1.0)
    hits = _wrrf(fts, vec, wf, wv, rrf_k, limit*(3 if spans else 1))
    if spans: hits = merge_spans(hits, gap)
    for h in hits[:limit]: h['breadcrumb'] = h.get('heading') or self.breadcrumb(h.get('node_id') or '', store, prefix)
    return hits[:limit]

def _wrrf(fts, vec, wf=1.0, wv=1.0, k=60, limit=50, id_key='rowid'):
    'Weighted RRF over the two legs. `rrf_merge` with a thumb on the scale.'
    scores = {}
    for lst, w in ((fts, wf), (vec, wv)):
        for rank, row in enumerate(lst or []):
            rid = row.get(id_key, id(row))
            if rid in scores: scores[rid]['_rrf_score'] += w/(k+rank)
            else: scores[rid] = dict(row, _rrf_score=w/(k+rank))
    return sorted(scores.values(), key=lambda r: -r['_rrf_score'])[:limit]

@patch
def sections(self:Database,
             q:str,               # query string
             emb:bytes,           # query embedding
             limit:int=5,         # sections to return
             per:int=3,           # snippets kept per section
             store:str='store',
             prefix:str=None,
             fanout:int=8,        # chunk hits gathered before rolling up
             score:str='max',     # how a section scores from its hits: max | mean | sum
             **kw                 # forwarded to doc_search
) -> list:
    '''Ranked *sections*, not chunks: hits grouped by node, each with a `read` handle.

    `score='max'` — a section is as relevant as its best evidence. The obvious alternative, summing
    the RRF mass of every hit in a node, reads well ("five weak hits in a chapter beat one strong
    hit in an appendix") and measures badly: it is a length prior in disguise. On 150 known-item
    queries over 486 pages of legislation, summing cost 0.07 MRR against `max` on verbatim queries
    and 0.16 on keyword-degraded ones, because long chapters outranked the precise article. `mean`
    is within noise of `max`; `sum` remains available for corpora of uniform section length.'''
    hits = self.doc_search(q, emb, limit=max(limit*fanout, 20), store=store, prefix=prefix, spans=False, **kw)
    g, agg = self.get_tree(store, prefix), {}
    for h in hits:
        nid = h.get('node_id')
        if not nid: continue
        a = agg.setdefault(nid, dict(node_id=nid, score=0.0, _sum=0.0, _n=0, snippets=[], pages=[]))
        s_ = h.get('_rrf_score', 0.0)
        a['_sum'] += s_; a['_n'] += 1
        a['score'] = (a['_sum'] if score == 'sum' else
                      a['_sum']/a['_n'] if score == 'mean' else max(a['score'], s_))
        if len(a['snippets']) < per: a['snippets'].append((h.get('content') or '')[:400])
        if h.get('page') is not None: a['pages'].append(h['page'])
    if not agg: return []
    nodes = {r['id']: r for r in g.nodes(where=_in('id', list(agg)))}
    out = []
    for nid, a in sorted(agg.items(), key=lambda kv: -kv[1]['score'])[:limit]:
        nd = nodes.get(nid) or {}
        out.append(dict(node_id=nid, title=nd.get('title', ''), score=a['score'],
                        breadcrumb=self.breadcrumb(nid, store, prefix), summary=nd.get('summary', ''),
                        pages=(min(a['pages']) if a['pages'] else nd.get('page_start'),
                               max(a['pages']) if a['pages'] else nd.get('page_end')),
                        nchunks=nd.get('nchunks', 0), snippets=a['snippets'],
                        read=f'read({nid!r})'))
    return out

## End to end

A three-chapter document, a deterministic hash embedder (so the notebook runs offline), and every
call in the module.

In [ ]:
import numpy as np, hashlib
from litesearch.core import database
from litesearch.graph import hash_embed

DOC = """# Saturn

Saturn is the slowest of the classical grahas.

## Sade Sati

Sade sati is the seven and a half year period when Saturn transits the twelfth, first and second
houses from the natal moon. It is counted in three phases of roughly thirty months each.

The middle phase, with Saturn over the moon itself, is held to be the heaviest.

## Remedies

Recitation on Saturdays is the common prescription. Donation of black sesame is another.

# Jupiter

Jupiter is the great benefic and moves through one sign each year.

## Transits

Jupiter's return to its natal sign happens roughly every twelve years."""

db  = database()
enc = lambda ts, **kw: hash_embed(ts, 256)
db.get_tree('store')
print(db.add_doc(DOC, title='Grahas', source='notes/grahas.md', kind='md', emb_fn=enc))

{'doc_id': '5a991f283775ae0d', 'title': 'Grahas', 'kind': 'md', 'nodes': 6, 'chunks': 5}


In [ ]:
# the tree came out of the headings, and every chunk knows which node it lives in
t = db.toc('Grahas')[0]
def show(n, d=0):
    print('  '*d + f"{n['title']:<14} chunks={n['nchunks']}  {n['id']}")
    for c in n.get('children', []): show(c, d+1)
show(t['tree'])
assert [c['title'] for c in t['tree']['children']] == ['Saturn', 'Jupiter']
assert db.breadcrumb(t['tree']['children'][0]['children'][0]['id']) == 'Grahas › Saturn › Sade Sati'

Grahas         chunks=0  5a991f283775ae0d#0
  Saturn         chunks=1  5a991f283775ae0d#1
    Sade Sati      chunks=1  5a991f283775ae0d#2
    Remedies       chunks=1  5a991f283775ae0d#3
  Jupiter        chunks=1  5a991f283775ae0d#4
    Transits       chunks=1  5a991f283775ae0d#5


In [ ]:
# read() returns the section, not a fragment -- and includes its children
sec = first(t['tree']['children'][0]['children'], lambda c: c['title'] == 'Sade Sati')
r = db.read(sec['id'])
print(r['breadcrumb'], '|', r['pages'], '|', len(r['text']), 'chars')
assert 'thirty months' in r['text'] and r['breadcrumb'].endswith('Sade Sati')

Grahas › Saturn › Sade Sati | (0, 0) | 266 chars


In [ ]:
# hybrid search, but every hit is placed in the document
q  = 'how long does the saturn transit last'
hits = db.doc_search(q, enc([q])[0].tobytes(), limit=3, dtype=np.float16)
for h in hits: print(f"{h['_rrf_score']:.4f}  {h['breadcrumb']:<28} {h['content'][:60]!r}")
assert hits and all(h['breadcrumb'] for h in hits)

# sections() rolls the same evidence up a level and hands back the next call
secs = db.sections(q, enc([q])[0].tobytes(), limit=2, dtype=np.float16)
for s in secs: print(f"{s['score']:.4f}  {s['breadcrumb']:<28} {s['read']}")
assert secs and secs[0]['read'].startswith('read(')

0.0167  Grahas › Saturn › Sade Sati  'Sade sati is the seven and a half year period when Saturn tr'
0.0164  Grahas › Saturn              'Saturn is the slowest of the classical grahas.'
0.0161  Grahas › Saturn › Remedies   'Recitation on Saturdays is the common prescription. Donation'
0.0167  Grahas › Saturn › Sade Sati  read('5a991f283775ae0d#2')
0.0164  Grahas › Saturn              read('5a991f283775ae0d#1')


In [ ]:
# adaptive fusion: a quoted phrase leans on FTS, an empty FTS leg hands the vote to vectors
assert adaptive_weights('"sade sati"', [1], [1])[0] > adaptive_weights('how long does it last', [1], [1])[0]
assert adaptive_weights('anything', [], [1]) == (0.0, 1.0)
assert adaptive_weights('anything', [1], []) == (1.0, 0.0)

# spans: two hits one page apart in the same node become one passage
_h = [dict(content='first half', node_id='d#1', page=3, _rrf_score=0.02, rowid=1),
      dict(content='second half', node_id='d#1', page=4, _rrf_score=0.01, rowid=2),
      dict(content='elsewhere',   node_id='d#9', page=40, _rrf_score=0.005, rowid=3)]
_m = merge_spans(_h)
assert len(_m) == 2 and _m[0]['_nspan'] == 2 and 'second half' in _m[0]['content'], _m
assert _m[0]['_rrf_score'] == 0.02, 'a merged span keeps its best score, it does not accumulate'

In [ ]:
# ingestion is content-addressed, and delete_doc leaves nothing behind
again = db.add_doc(DOC, title='Grahas', source='notes/grahas.md', emb_fn=enc)
assert 'skipped' in again, again
did = t['doc_id']
n_before = len(db.t.store())
db.delete_doc(did)
assert len(db.t.store()) == 0 and db.toc() == [] and n_before > 0
print(f'{n_before} chunks removed with the document')

5 chunks removed with the document


## What this does not do yet

- **No LLM anywhere.** `summarize=` takes a callable and `build_tree` can be replaced wholesale;
  an LLM-written summary per node is the single highest-value place to spend tokens, because it is
  what `toc()` shows an agent that is deciding where to read.
- **Chunking is `chunk_markdown`.** A cost-model splitter (sentences → chunklets → semantic merge,
  as in RAGLite) produces better boundaries on prose. The seam is `chunker=`.
- **Late chunking is available but not wired in here.** `litesearch.utils.LateChunkFastEncode`
  contextualises a node's chunks jointly; passing one as `emb_fn` already works, but nothing yet
  batches per node to make it worthwhile.
- **Chunking is where the measured gains are.** On the evaluation in `nbs/07_doc_eval.ipynb`,
  moving from page-sized chunks to node-scoped ones lifted MRR from 0.330 to 0.522 on degraded
  queries — more than every other feature here combined. The cost-model splitter is the next
  thing to port, not a refinement of the ranking.

In [ ]:
import tempfile
_d = Path(tempfile.mkdtemp())
(_d/'ratios.md').write_text('# Ratios\n\n## Liquidity\n\nCurrent ratio is current assets over current liabilities.')
(_d/'notes.txt').write_text('A plain text file with no headings at all, ingested as one windowed node.')
_fdb = database()
_fdb.get_tree('store')
print(_fdb.add_dir(_d, emb_fn=enc))
assert {d['title'] for d in _fdb.toc()} == {'ratios', 'notes'}
_rt = _fdb.toc('ratios')[0]['tree']
assert [n['title'] for n in _rt['children']] == ['Ratios']            # the `# Ratios` heading
assert [n['title'] for n in _rt['children'][0]['children']] == ['Liquidity']
assert _fdb.add_file(_d/'ratios.md', emb_fn=enc).get('skipped'), 're-ingest is a no-op'
assert 'current liabilities' in _fdb.read(_fdb.toc('ratios')[0]['tree']['id'])['text']

[{'doc_id': 'acdecfddcad969a0', 'title': 'notes', 'kind': 'txt', 'nodes': 2, 'chunks': 1}, {'doc_id': 'fca03a0aa3f13fff', 'title': 'ratios', 'kind': 'md', 'nodes': 3, 'chunks': 1}]


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()